# 01 - Feature Engineering: Customer Financial Profile

## FinPulse AI — Plataforma de Inteligência Financeira para Open Finance

Este notebook tem como objetivo transformar a tabela analítica `mart_customer_financial_profile`, criada com dbt, em uma base de features pronta para modelos de IA, análises financeiras e aplicações de negócio.

A tabela original já consolida informações de clientes, contas, cartões, empréstimos e transações. A partir dela, serão criadas variáveis derivadas para apoiar os próximos módulos do projeto:

- Financial Health Score
- Churn Risk
- Detecção de anomalias financeiras
- Recomendações personalizadas
- Dashboard em Streamlit
- API com FastAPI
- Assistente inteligente com IA Generativa

O foco desta etapa é preparar uma base confiáutilizando comportamento passado para prever inatividade futura.

## Notebook Plan

1. Setup and database connection
2. Load customer financial mart
3. Initial data validation
4. Date treatment
5. Customer relationship features
6. Account and balance features
7. Card features
8. Loan and credit exposure features
9. Transaction behavior features
10. Engagement and risk features
11. Feature validation
12. Export processed feature dataset
13. Conclusions and next steps

## 1. Setup and Database Connection

In this step, we import the required Python libraries and create a connection with the PostgreSQL database running inside Docker.

The notebook connects to the same database where the dbt mart `mart_customer_financial_profile` was created.

In [1]:

pip install psycopg2-binary 


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

DB_USER = "finpulse_user"
DB_PASSWORD = "finpulse_password"
DB_HOST = "postgres"
DB_PORT = "5432"
DB_NAME = "finpulse"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

## 2. Load Customer Financial Mart

The feature engineering process starts from the dbt mart `mart_customer_financial_profile`.

This mart contains one row per customer and consolidates customer profile, accounts, cards, loans and transaction behavior.

In [3]:
query = """
SELECT *
FROM dbt.mart_customer_financial_profile
"""

df = pd.read_sql(query, engine)

df.head(1000)

,customer_id,first_name,last_name,email,city,credit_score,customer_created_at,total_accounts,total_balance_usd,avg_balance_usd,...,total_transactions,total_transaction_amount_usd,avg_transaction_amount_usd,min_transaction_amount_usd,max_transaction_amount_usd,first_transaction_date,last_transaction_date,active_transaction_days,merchant_diversity,merchant_city_diversity
0,CUS002V4AVJO5UQ,Briana,Pugh,lisaspears@example.org,Hobbston,510,2019-12-03,3,309600.73,103200.243333,...,44,193782.79,4404.154318,310.03,9828.92,2019-05-11 07:18:31,2025-11-14 07:49:06,44,44,44
1,CUS0031SK48C4X5,Julie,Harding,greglee@example.org,Lovetown,540,2022-11-23,2,207173.59,103586.795000,...,25,115131.11,4605.244400,351.49,9507.20,2019-11-28 07:55:26,2025-12-22 22:38:56,25,25,25
2,CUS003E82EMAYB3,Lindsey,Hamilton,kburgess@example.com,Port Stephanie,739,2025-09-11,2,238950.91,119475.455000,...,19,92845.63,4886.612105,198.75,9948.82,2019-01-01 11:43:44,2025-11-11 22:28:10,19,19,19
3,CUS0079NNF0ML3Z,Jacqueline,Saunders,renee68@example.org,Jessemouth,678,2023-05-01,1,173017.15,173017.150000,...,10,46577.23,4657.723000,132.77,9054.88,2019-10-30 12:06:22,2025-06-23 18:11:01,10,10,10
4,CUS008083HIW01T,Stacey,Walker,moorecheryl@example.com,North Mary,834,2025-10-14,0,0.00,0.000000,...,0,0.00,0.000000,0.00,0.00,NaT,NaT,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,CUS1GUVCPYYC6UF,Randy,Hale,william27@example.com,North John,340,2024-02-26,1,104163.53,104163.530000,...,12,57918.31,4826.525833,173.34,9878.12,2019-10-17 14:15:25,2025-08-14 00:05:36,12,12,11
996,CUS1GWSFLZIOUPA,Madeline,Roberson,hsweeney@example.net,Evansview,355,2022-10-02,3,325818.85,108606.283333,...,39,178264.82,4570.892821,10.06,9578.92,2019-01-13 22:42:19,2025-12-09 06:43:32,39,39,39
997,CUS1GZGPUJP1XGC,Karen,Morgan,ymurray@example.com,Mosleyton,730,2023-04-20,1,4403.28,4403.280000,...,13,62956.82,4842.832308,213.57,9861.50,2019-05-13 15:55:12,2025-05-13 20:40:50,13,13,13
998,CUS1H1L6P4CF0AY,Sharon,Williamson,brittney81@example.org,North Glen,447,2024-08-02,0,0.00,0.000000,...,0,0.00,0.000000,0.00,0.00,NaT,NaT,0,0,0


## 3. Initial Data Validation

Before creating new features, we validate the structure of the dataset.

This step checks the number of rows and columns, duplicated customers, missing values and general data types.

In [4]:
df.shape

(50000, 45)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 45 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_id                   50000 non-null  object        
 1   first_name                    50000 non-null  object        
 2   last_name                     50000 non-null  object        
 3   email                         50000 non-null  object        
 4   city                          50000 non-null  object        
 5   credit_score                  50000 non-null  int64         
 6   customer_created_at           50000 non-null  datetime64[ns]
 7   total_accounts                50000 non-null  int64         
 8   total_balance_usd             50000 non-null  float64       
 9   avg_balance_usd               50000 non-null  float64       
 10  min_balance_usd               50000 non-null  float64       
 11  max_balance_usd             

In [6]:
missing_values = df.isna().sum().sort_values(ascending=False)

missing_values[missing_values > 0]

last_loan_start_date          27414
first_loan_start_date         27414
first_card_expiration_date    16496
last_card_expiration_date     16496
last_transaction_date         11162
first_transaction_date        11162
last_account_open_date        11162
first_account_open_date       11162
dtype: int64

In [7]:
df.duplicated(subset=["customer_id"]).sum()

0

In [8]:
df.loc[df["has_loan"] == 0, ["first_loan_start_date", "last_loan_start_date"]].isna().sum()

first_loan_start_date    27414
last_loan_start_date     27414
dtype: int64

In [9]:
df.loc[df["has_card"] == 0, ["first_card_expiration_date", "last_card_expiration_date"]].isna().sum()

first_card_expiration_date    16496
last_card_expiration_date     16496
dtype: int64

In [10]:
df.loc[df["has_transaction"] == 0, ["first_transaction_date", "last_transaction_date"]].isna().sum()

first_transaction_date    11162
last_transaction_date     11162
dtype: int64

## 4. Date Treatment

In this step, we convert date columns to datetime format and define a reference date for time-based feature engineering.

Time-based features are important for understanding customer tenure, account relationship, transaction recency, loan recency and card expiration behavior.

These features will later support churn risk, financial health score and engagement analysis.

In [11]:
date_columns = [
    "customer_created_at",
    "first_account_open_date",
    "last_account_open_date",
    "first_card_expiration_date",
    "last_card_expiration_date",
    "first_loan_start_date",
    "last_loan_start_date",
    "first_transaction_date",
    "last_transaction_date"
]

date_columns

['customer_created_at',
 'first_account_open_date',
 'last_account_open_date',
 'first_card_expiration_date',
 'last_card_expiration_date',
 'first_loan_start_date',
 'last_loan_start_date',
 'first_transaction_date',
 'last_transaction_date']

In [12]:
date_columns = [
    "customer_created_at",
    "first_account_open_date",
    "last_account_open_date",
    "first_card_expiration_date",
    "last_card_expiration_date",
    "first_loan_start_date",
    "last_loan_start_date",
    "first_transaction_date",
    "last_transaction_date"
]

date_columns

['customer_created_at',
 'first_account_open_date',
 'last_account_open_date',
 'first_card_expiration_date',
 'last_card_expiration_date',
 'first_loan_start_date',
 'last_loan_start_date',
 'first_transaction_date',
 'last_transaction_date']

In [13]:
for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

In [14]:
df[date_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   customer_created_at         50000 non-null  datetime64[ns]
 1   first_account_open_date     38838 non-null  datetime64[ns]
 2   last_account_open_date      38838 non-null  datetime64[ns]
 3   first_card_expiration_date  33504 non-null  datetime64[ns]
 4   last_card_expiration_date   33504 non-null  datetime64[ns]
 5   first_loan_start_date       22586 non-null  datetime64[ns]
 6   last_loan_start_date        22586 non-null  datetime64[ns]
 7   first_transaction_date      38838 non-null  datetime64[ns]
 8   last_transaction_date       38838 non-null  datetime64[ns]
dtypes: datetime64[ns](9)
memory usage: 3.4 MB


In [15]:
snapshot_date = df["last_transaction_date"].max()

snapshot_date

Timestamp('2025-12-31 23:55:27')

### Data de referência do perfil

`snapshot_date` representa a data de referência utilizada para construir o retrato financeiro atual dos clientes.

As variáveis temporais deste notebook utilizam todo o histórico disponível até essa data. Portanto, elas são adequadas para análises descritivas e indicadores atuais, mas não constituem, por si só, uma base temporal de treinamento.

Na etapa de Machine Learning, as features serão recalculadas em diferentes datas de corte, utilizando somente informações disponíveis até cada data.

In [16]:
date_ranges = df[date_columns].agg(["min", "max"]).T

date_ranges

,min,max
customer_created_at,2019-01-01 00:00:00,2025-12-31 00:00:00
first_account_open_date,2019-01-01 00:00:00,2025-12-31 00:00:00
last_account_open_date,2019-01-01 00:00:00,2025-12-31 00:00:00
first_card_expiration_date,2025-01-01 00:00:00,2032-12-31 00:00:00
last_card_expiration_date,2025-01-01 00:00:00,2032-12-31 00:00:00
first_loan_start_date,2019-01-01 00:00:00,2025-12-31 00:00:00
last_loan_start_date,2019-01-01 00:00:00,2025-12-31 00:00:00
first_transaction_date,2019-01-01 00:00:23,2024-08-25 07:57:51
last_transaction_date,2020-12-13 05:54:34,2025-12-31 23:55:27


### Temporal Data Quality Audit

Before creating time-based features, we validate the chronological consistency between customers, accounts, loans and transactions.

These checks identify records where:

- an account was opened before the customer was created;
- a loan started before the customer was created;
- a transaction occurred before the corresponding account was opened;
- a transaction occurred before the customer was created.

The results will determine whether temporal corrections or filtering are required before predictive modeling.

In [17]:
temporal_quality_query = """
SELECT
    'accounts_before_customer' AS quality_check,
    COUNT(*) AS invalid_records
FROM dbt.stg_accounts a
INNER JOIN dbt.stg_customers c
    ON a.customer_id = c.customer_id
WHERE a.open_date < c.created_at::date

UNION ALL

SELECT
    'loans_before_customer' AS quality_check,
    COUNT(*) AS invalid_records
FROM dbt.stg_loans l
INNER JOIN dbt.stg_customers c
    ON l.customer_id = c.customer_id
WHERE l.start_date < c.created_at::date

UNION ALL

SELECT
    'transactions_before_account' AS quality_check,
    COUNT(*) AS invalid_records
FROM dbt.stg_transactions t
INNER JOIN dbt.stg_accounts a
    ON t.account_id = a.account_id
WHERE t.transaction_date::date < a.open_date

UNION ALL

SELECT
    'transactions_before_customer' AS quality_check,
    COUNT(*) AS invalid_records
FROM dbt.stg_transactions t
INNER JOIN dbt.stg_accounts a
    ON t.account_id = a.account_id
INNER JOIN dbt.stg_customers c
    ON a.customer_id = c.customer_id
WHERE t.transaction_date < c.created_at;
"""

temporal_quality = pd.read_sql(
    temporal_quality_query,
    engine
)

display(temporal_quality)

,quality_check,invalid_records
0,loans_before_customer,15069
1,accounts_before_customer,37408
2,transactions_before_account,501669
3,transactions_before_customer,502033


### Impact of temporal inconsistencies

The audit identified systematic chronological inconsistencies in the synthetic source dataset.

Approximately half of the accounts, loans and transactions violate the expected customer lifecycle chronology. This indicates that the source dates were likely generated independently.

The records will not be automatically deleted or modified because doing so would remove a substantial portion of the dataset or fabricate historical information.

The following analysis measures how these inconsistencies affect the customer-level financial profile.

In [18]:
customer_temporal_checks = [
    (
        "first_account_before_customer",
        df["first_account_open_date"] < df["customer_created_at"],
        df["total_accounts"] > 0
    ),
    (
        "last_account_before_customer",
        df["last_account_open_date"] < df["customer_created_at"],
        df["total_accounts"] > 0
    ),
    (
        "first_loan_before_customer",
        df["first_loan_start_date"] < df["customer_created_at"],
        df["has_loan"] == 1
    ),
    (
        "last_loan_before_customer",
        df["last_loan_start_date"] < df["customer_created_at"],
        df["has_loan"] == 1
    ),
    (
        "first_transaction_before_customer",
        df["first_transaction_date"] < df["customer_created_at"],
        df["has_transaction"] == 1
    ),
    (
        "last_transaction_before_customer",
        df["last_transaction_date"] < df["customer_created_at"],
        df["has_transaction"] == 1
    )
]

customer_temporal_impact = []

for check_name, invalid_condition, population_condition in customer_temporal_checks:
    affected_customers = int(
        (invalid_condition & population_condition)
        .fillna(False)
        .sum()
    )

    evaluated_customers = int(
        population_condition
        .fillna(False)
        .sum()
    )

    affected_percentage = (
        affected_customers / evaluated_customers * 100
        if evaluated_customers > 0
        else 0
    )

    customer_temporal_impact.append({
        "quality_check": check_name,
        "affected_customers": affected_customers,
        "evaluated_customers": evaluated_customers,
        "affected_percentage": round(affected_percentage, 2)
    })

customer_temporal_impact = pd.DataFrame(
    customer_temporal_impact
)

display(customer_temporal_impact)

,quality_check,affected_customers,evaluated_customers,affected_percentage
0,first_account_before_customer,24033,38838,61.88
1,last_account_before_customer,14792,38838,38.09
2,first_loan_before_customer,12475,22586,55.23
3,last_loan_before_customer,10228,22586,45.28
4,first_transaction_before_customer,36870,38838,94.93
5,last_transaction_before_customer,1922,38838,4.95


## 5. Time-Based Features

Due to chronological inconsistencies in the synthetic source data, account opening dates and loan start dates will not be used to create relationship-duration or loan-recency features.

The customer creation date is interpreted only as the age of the customer record.

Transaction recency is calculated only when the customer has transactions and the last transaction occurs on or after customer creation.

Customers without transactions are treated as not activated, rather than automatically classified as churn.

### Features created

- `customer_record_age_days`: age of the customer record;
- `days_since_last_transaction`: valid transaction recency;
- `days_until_latest_card_expiration`: time until the latest card expiration;
- `never_transacted_flag`: customer has no observed transactions;
- `transaction_timeline_invalid_flag`: last transaction occurred before customer creation;
- `transaction_recency_valid_flag`: transaction recency can be safely interpreted.ss meaning of missing activity or missing products.

In [19]:
df["customer_record_age_days"] = (
    snapshot_date - df["customer_created_at"]
).dt.days

df["never_transacted_flag"] = (
    df["has_transaction"] == 0
).astype("int8")

df["transaction_timeline_invalid_flag"] = (
    (df["has_transaction"] == 1) &
    (
        df["last_transaction_date"] <
        df["customer_created_at"]
    )
).astype("int8")

df["transaction_recency_valid_flag"] = (
    (df["has_transaction"] == 1) &
    (df["transaction_timeline_invalid_flag"] == 0) &
    df["last_transaction_date"].notna()
).astype("int8")

df["days_since_last_transaction"] = (
    snapshot_date - df["last_transaction_date"]
).dt.days

df.loc[
    df["transaction_recency_valid_flag"] == 0,
    "days_since_last_transaction"
] = np.nan

df["days_until_latest_card_expiration"] = (
    df["last_card_expiration_date"] - snapshot_date
).dt.days

In [20]:
time_feature_quality = pd.DataFrame({
    "metric": [
        "customers_without_transactions",
        "customers_with_invalid_transaction_timeline",
        "customers_with_valid_transaction_recency"
    ],
    "customers": [
        int(df["never_transacted_flag"].sum()),
        int(df["transaction_timeline_invalid_flag"].sum()),
        int(df["transaction_recency_valid_flag"].sum())
    ]
})

display(time_feature_quality)

,metric,customers
0,customers_without_transactions,11162
1,customers_with_invalid_transaction_timeline,1922
2,customers_with_valid_transaction_recency,36916


In [21]:
time_features = [
    "customer_record_age_days",
    "days_since_last_transaction",
    "days_until_latest_card_expiration",
    "never_transacted_flag",
    "transaction_timeline_invalid_flag",
    "transaction_recency_valid_flag"
]

df[time_features].describe().T

,count,mean,std,min,25%,50%,75%,max
customer_record_age_days,50000.0,1271.437160,741.122979,0.0,627.0,1270.0,1917.25,2556.0
days_since_last_transaction,36916.0,118.972857,139.862137,0.0,28.0,71.0,157.00,1805.0
days_until_latest_card_expiration,33504.0,1656.444633,745.681027,-365.0,1193.0,1868.0,2270.25,2556.0
never_transacted_flag,50000.0,0.223240,0.416422,0.0,0.0,0.0,0.00,1.0
transaction_timeline_invalid_flag,50000.0,0.038440,0.192258,0.0,0.0,0.0,0.00,1.0
transaction_recency_valid_flag,50000.0,0.738320,0.439554,0.0,0.0,1.0,1.00,1.0


In [22]:
df[time_features].isna().sum()

customer_record_age_days                 0
days_since_last_transaction          13084
days_until_latest_card_expiration    16496
never_transacted_flag                    0
transaction_timeline_invalid_flag        0
transaction_recency_valid_flag           0
dtype: int64

### Temporal feature decision

The customer, account and loan dates in the synthetic dataset are not chronologically consistent.

Because these inconsistencies affect a substantial portion of the records:

- account opening dates will not be used to measure relationship duration;
- loan start dates will not be used to measure loan recency;
- the customer creation date will be interpreted only as the age of the customer record;
- transaction recency will be calculated only when the last transaction occurs on or after customer creation;
- customers without transactions will be classified as not activated, rather than automatically classified as churn.

No source dates will be deleted or artificially modified.

In [23]:
df[time_features].isna().sum()

customer_record_age_days                 0
days_since_last_transaction          13084
days_until_latest_card_expiration    16496
never_transacted_flag                    0
transaction_timeline_invalid_flag        0
transaction_recency_valid_flag           0
dtype: int64

## 6. Account and Balance Features

In this step, we create features based on customer accounts and account balances.

These features help describe the customer's banking relationship, balance distribution and account structure.

They will be useful for financial health scoring, customer segmentation, credit risk analysis and recommendation logic.

### Features created in this step

- `balance_per_account`  
  Average balance per account. It helps measure how much balance the customer holds for each account relationship.

- `balance_concentration_ratio`  
  Ratio between the highest account balance and the total account balance.  
  This feature indicates whether the customer's money is concentrated in one account or distributed across multiple accounts.

- `has_savings_account`  
  Indicates whether the customer has at least one savings account.

- `has_business_account`  
  Indicates whether the customer has at least one business account.

- `has_checking_account`  
  Indicates whether the customer has at least one checking account.

These features improve the customer financial profile by transforming raw account counts and balances into more interpretable financial behavior indicators.

In [24]:
df["balance_per_account"] = np.where(
    df["total_accounts"] > 0,
    df["total_balance_usd"] / df["total_accounts"],
    0
)

df["balance_concentration_ratio"] = np.where(
    df["total_balance_usd"] > 0,
    df["max_balance_usd"] / df["total_balance_usd"],
    0
)

df["has_savings_account"] = np.where(df["savings_accounts"] > 0, 1, 0)
df["has_business_account"] = np.where(df["business_accounts"] > 0, 1, 0)
df["has_checking_account"] = np.where(df["checking_accounts"] > 0, 1, 0)

In [25]:
account_features = [
    "balance_per_account",
    "balance_concentration_ratio",
    "has_savings_account",
    "has_business_account",
    "has_checking_account"
]

df[account_features].describe().T

,count,mean,std,min,25%,50%,75%,max
balance_per_account,50000.0,77853.019510,58711.957275,0.0,14971.862500,82305.888750,122579.27,199968.62
balance_concentration_ratio,50000.0,0.599272,0.382617,0.0,0.353216,0.633066,1.00,1.00
has_savings_account,50000.0,0.395600,0.488984,0.0,0.000000,0.000000,1.00,1.00
has_business_account,50000.0,0.394320,0.488709,0.0,0.000000,0.000000,1.00,1.00
has_checking_account,50000.0,0.395400,0.488941,0.0,0.000000,0.000000,1.00,1.00


In [26]:
df[account_features].isna().sum()

balance_per_account            0
balance_concentration_ratio    0
has_savings_account            0
has_business_account           0
has_checking_account           0
dtype: int64

In [27]:
df[
    ["has_savings_account", "has_business_account", "has_checking_account"]
].sum().sort_values(ascending=False)

has_savings_account     19780
has_checking_account    19770
has_business_account    19716
dtype: int64

### Account Feature Validation

The account and balance features were successfully created with no missing values.

The `balance_concentration_ratio` ranges from 0 to 1, which confirms that the feature was calculated correctly. A value closer to 1 means the customer's balance is highly concentrated in a single account, while lower values indicate a more distributed balance across accounts.

The account type flags show whether each customer has checking, savings or business accounts. These indicators will be useful for customer segmentation, financial health scoring, recommendation rules and dashboard analysis.

In [28]:
df.head()

,customer_id,first_name,last_name,email,city,credit_score,customer_created_at,total_accounts,total_balance_usd,avg_balance_usd,...,never_transacted_flag,transaction_timeline_invalid_flag,transaction_recency_valid_flag,days_since_last_transaction,days_until_latest_card_expiration,balance_per_account,balance_concentration_ratio,has_savings_account,has_business_account,has_checking_account
0,CUS002V4AVJO5UQ,Briana,Pugh,lisaspears@example.org,Hobbston,510,2019-12-03,3,309600.73,103200.243333,...,0,0,1,47.0,2529.0,103200.243333,0.539187,1,1,0
1,CUS0031SK48C4X5,Julie,Harding,greglee@example.org,Lovetown,540,2022-11-23,2,207173.59,103586.795000,...,0,0,1,9.0,2061.0,103586.795000,0.870969,1,0,1
2,CUS003E82EMAYB3,Lindsey,Hamilton,kburgess@example.com,Port Stephanie,739,2025-09-11,2,238950.91,119475.455000,...,0,0,1,50.0,2529.0,119475.455000,0.734799,1,0,1
3,CUS0079NNF0ML3Z,Jacqueline,Saunders,renee68@example.org,Jessemouth,678,2023-05-01,1,173017.15,173017.150000,...,0,0,1,191.0,2263.0,173017.150000,1.000000,0,1,0
4,CUS008083HIW01T,Stacey,Walker,moorecheryl@example.com,North Mary,834,2025-10-14,0,0.00,0.000000,...,1,0,0,NaN,NaN,0.000000,0.000000,0,0,0


## 7. Card Features

In this step, we create features based on the customer's card relationship.

The original mart already contains the number of cards, credit cards and debit cards per customer. In this section, we transform these values into more interpretable indicators.

These features can support customer segmentation, churn risk analysis, card renewal recommendations and personalized financial offers.

### Features created in this step

- `has_credit_card`  
  Indicates whether the customer has at least one credit card.

- `has_debit_card`  
  Indicates whether the customer has at least one debit card.

- `credit_card_ratio`  
  Ratio of credit cards over total cards.  
  This feature helps identify customers more connected to credit products.

- `debit_card_ratio`  
  Ratio of debit cards over total cards.  
  This feature helps identify customers more connected to transactional or checking behavior.

- `card_product_diversity`  
  Number of different card product types used by the customer.  
  A customer with both credit and debit cards has higher card relationship diversity.

These features help describe how strong the customer's card relationship is with the bank.

In [29]:
df["has_credit_card"] = np.where(df["credit_cards"] > 0, 1, 0)

df["has_debit_card"] = np.where(df["debit_cards"] > 0, 1, 0)

df["credit_card_ratio"] = np.where(
    df["total_cards"] > 0,
    df["credit_cards"] / df["total_cards"],
    0
)

df["debit_card_ratio"] = np.where(
    df["total_cards"] > 0,
    df["debit_cards"] / df["total_cards"],
    0
)

df["card_product_diversity"] = (
    df["has_credit_card"] + df["has_debit_card"]
)

In [30]:
card_features = [
    "has_credit_card",
    "has_debit_card",
    "credit_card_ratio",
    "debit_card_ratio",
    "card_product_diversity"
]

df[card_features].describe().T

,count,mean,std,min,25%,50%,75%,max
has_credit_card,50000.0,0.517360,0.499704,0.0,0.0,1.00,1.000,1.0
has_debit_card,50000.0,0.522420,0.499502,0.0,0.0,1.00,1.000,1.0
credit_card_ratio,50000.0,0.332116,0.372376,0.0,0.0,0.25,0.600,1.0
debit_card_ratio,50000.0,0.337964,0.374958,0.0,0.0,0.25,0.625,1.0
card_product_diversity,50000.0,1.039780,0.835495,0.0,0.0,1.00,2.000,2.0


In [31]:
df[card_features].isna().sum()

has_credit_card           0
has_debit_card            0
credit_card_ratio         0
debit_card_ratio          0
card_product_diversity    0
dtype: int64

In [32]:
df["card_product_diversity"].value_counts().sort_index()

card_product_diversity
0    16496
1    15019
2    18485
Name: count, dtype: int64

### Card Feature Validation

The card features were successfully created with no missing values.

The `credit_card_ratio` and `debit_card_ratio` range from 0 to 1, confirming that the ratio calculations are valid.

The `card_product_diversity` feature ranges from 0 to 2:
- 0 means the customer has no card.
- 1 means the customer has either credit or debit card.
- 2 means the customer has both credit and debit cards.

These features are useful for understanding product adoption, customer engagement and potential recommendation opportunities.

In [33]:
df.shape

(50000, 61)

In [34]:
preview_columns = [
    "customer_id",
    "first_name",
    "city",
    "credit_score",

    "total_accounts",
    "total_balance_usd",
    "balance_per_account",
    "balance_concentration_ratio",
    "has_savings_account",
    "has_business_account",
    "has_checking_account",

    "has_card",
    "total_cards",
    "has_credit_card",
    "has_debit_card",
    "credit_card_ratio",
    "debit_card_ratio",
    "card_product_diversity",

    "customer_record_age_days",
    "days_since_last_transaction",
    "never_transacted_flag",
    "transaction_timeline_invalid_flag",
    "transaction_recency_valid_flag"
]

df[preview_columns].head(10)

,customer_id,first_name,city,credit_score,total_accounts,total_balance_usd,balance_per_account,balance_concentration_ratio,has_savings_account,has_business_account,...,has_credit_card,has_debit_card,credit_card_ratio,debit_card_ratio,card_product_diversity,customer_record_age_days,days_since_last_transaction,never_transacted_flag,transaction_timeline_invalid_flag,transaction_recency_valid_flag
0,CUS002V4AVJO5UQ,Briana,Hobbston,510,3,309600.73,103200.243333,0.539187,1,1,...,1,1,0.500000,0.500000,2,2220,47.0,0,0,1
1,CUS0031SK48C4X5,Julie,Lovetown,540,2,207173.59,103586.795000,0.870969,1,0,...,1,0,1.000000,0.000000,1,1134,9.0,0,0,1
2,CUS003E82EMAYB3,Lindsey,Port Stephanie,739,2,238950.91,119475.455000,0.734799,1,0,...,1,1,0.666667,0.333333,2,111,50.0,0,0,1
3,CUS0079NNF0ML3Z,Jacqueline,Jessemouth,678,1,173017.15,173017.150000,1.000000,0,1,...,1,1,0.333333,0.666667,2,975,191.0,0,0,1
4,CUS008083HIW01T,Stacey,North Mary,834,0,0.00,0.000000,0.000000,0,0,...,0,0,0.000000,0.000000,0,78,NaN,1,0,0
5,CUS00D9DCE5I25S,Phillip,Deborahville,489,2,217603.95,108801.975000,0.536324,0,1,...,1,1,0.500000,0.500000,2,287,0.0,0,0,1
6,CUS00EX3A3P13EM,Billy,East Thomas,736,3,160748.90,53582.966667,0.821821,1,0,...,1,1,0.333333,0.666667,2,2315,32.0,0,0,1
7,CUS00FNFWDFF1CF,Kelly,South Codyborough,655,3,441643.01,147214.336667,0.409402,1,1,...,1,1,0.200000,0.800000,2,1229,17.0,0,0,1
8,CUS00FSPOJCW0NC,Darlene,Georgetown,590,1,24169.71,24169.710000,1.000000,1,0,...,0,0,0.000000,0.000000,0,586,115.0,0,0,1
9,CUS00IDX2LVEG7J,John,East Jacobchester,538,0,0.00,0.000000,0.000000,0,0,...,0,0,0.000000,0.000000,0,821,NaN,1,0,0


## 8. Loan and Credit Exposure Features

In this step, we create features related to customer loans and credit exposure.

The original mart contains loan counts, loan amounts and interest rate statistics. In this section, we transform these values into features that help measure credit usage, debt exposure and potential financial risk.

These features will support financial health scoring, credit risk analysis, churn modeling and recommendation logic.

### Features created in this step

- `loan_to_balance_ratio`  
  Ratio between the customer's total loan amount and total account balance.  
  This feature helps measure how large the customer's loan exposure is compared to their available balance.

- `avg_loan_per_account`  
  Average loan amount per account.  
  This feature helps normalize loan exposure by the customer's number of accounts.

- `loan_intensity`  
  Ratio between the number of loans and the number of accounts.  
  This feature indicates how frequently the customer uses loan products relative to their account relationship.

- `high_interest_flag`  
  Indicates whether the customer's average interest rate is above the 75th percentile among customers with loans.  
  This is a data-driven threshold used to identify customers with relatively high credit costs within the dataset. It is not a regulatory rule, but an initial segmentation criterion for risk analysis.

- `has_high_credit_exposure`  
  Indicates whether the customer's loan-to-balance ratio is high.  
  This feature can support credit risk and financial health analysis.

These features help describe how exposed the customer is to credit products and whether that exposure may represent financial pressure.

In [35]:
high_interest_threshold = df.loc[
    df["has_loan"] == 1,
    "avg_interest_rate"
].quantile(0.75)

high_interest_threshold

11.295

In [36]:
df["debt_without_balance_flag"] = (
    (df["total_loan_amount"] > 0) &
    (df["total_balance_usd"] <= 0)
).astype("int8")

df["loan_to_balance_ratio"] = np.select(
    [
        df["total_loan_amount"] == 0,
        (
            (df["total_loan_amount"] > 0) &
            (df["total_balance_usd"] > 0)
        )
    ],
    [
        0,
        (
            df["total_loan_amount"] /
            df["total_balance_usd"]
        )
    ],
    default=np.nan
)

df["avg_loan_per_account"] = np.select(
    [
        df["total_loan_amount"] == 0,
        df["total_accounts"] > 0
    ],
    [
        0,
        (
            df["total_loan_amount"] /
            df["total_accounts"]
        )
    ],
    default=np.nan
)

df["loan_intensity"] = np.select(
    [
        df["total_loans"] == 0,
        df["total_accounts"] > 0
    ],
    [
        0,
        (
            df["total_loans"] /
            df["total_accounts"]
        )
    ],
    default=np.nan
)

df["high_interest_flag"] = (
    (df["has_loan"] == 1) &
    (df["avg_interest_rate"] > high_interest_threshold)
).astype("int8")

valid_credit_exposure = df.loc[
    (df["has_loan"] == 1) &
    (df["total_balance_usd"] > 0),
    "loan_to_balance_ratio"
]

credit_exposure_threshold = (
    valid_credit_exposure.quantile(0.75)
)

df["has_high_credit_exposure"] = (
    (
        df["loan_to_balance_ratio"] >=
        credit_exposure_threshold
    ) |
    (df["debt_without_balance_flag"] == 1)
).astype("int8")

In [37]:
df["relationship_depth"] = (
    df["has_checking_account"] +
    df["has_savings_account"] +
    df["has_business_account"] +
    df["has_card"] +
    df["has_loan"] +
    df["has_transaction"]
)

df["product_diversity_score"] = (
    df["has_checking_account"] +
    df["has_savings_account"] +
    df["has_business_account"] +
    df["has_credit_card"] +
    df["has_debit_card"] +
    df["has_loan"]
)

In [38]:
df["debt_pressure_flag"] = (
    (df["has_high_credit_exposure"] == 1) |
    (df["high_interest_flag"] == 1) |
    (df["debt_without_balance_flag"] == 1)
).astype("int8")

df["is_financially_active"] = np.where(
    (df["has_transaction"] == 1) &
    (df["days_since_last_transaction"] <= 90),
    1,
    0
)

df["low_engagement_flag"] = np.where(
    (df["has_transaction"] == 0) |
    (df["days_since_last_transaction"] > 180),
    1,
    0
)

In [39]:
balance_p75 = df["total_balance_usd"].quantile(0.75)

df["financial_resilience_proxy"] = np.where(
    (df["total_balance_usd"] >= balance_p75) &
    (df["debt_pressure_flag"] == 0) &
    (df["is_financially_active"] == 1),
    1,
    0
)

### Interpretação das Features de Empréstimos e Exposição a Crédito

As features criadas nesta seção ajudam a medir a exposição do cliente a crédito, o comportamento em relação a empréstimos e possíveis sinais de pressão financeira.

Esses indicadores serão usados principalmente na Camada 5 do projeto, especialmente nos módulos de:

- Financial Health Score
- Análise de risco financeiro
- Risco de churn
- Recomendações personalizadas
- Alertas e insights no dashboard

---

### `loan_to_balance_ratio`

Fórmula:

`total_loan_amount / total_balance_usd`

Essa feature mede o tamanho da dívida do cliente em relação ao saldo total disponível em suas contas.

Exemplo:

- `total_loan_amount = 80.000`
- `total_balance_usd = 100.000`
- `loan_to_balance_ratio = 0.80`

Isso significa que o valor total dos empréstimos representa 80% do saldo total do cliente.

Quanto maior esse indicador, maior pode ser a exposição do cliente a crédito. Essa feature ajuda a identificar clientes que podem precisar de maior atenção em análises de risco financeiro.

---

### `avg_loan_per_account`

Fórmula:

`total_loan_amount / total_accounts`

Essa feature normaliza o valor total dos empréstimos pela quantidade de contas do cliente.

Ela ajuda a comparar clientes com estruturas diferentes. Por exemplo, um cliente com alto valor de empréstimo e apenas uma conta pode ter um perfil diferente de um cliente com o mesmo valor de empréstimo distribuído em várias contas.

Essa feature pode apoiar o score financeiro, a segmentação de clientes e análises de exposição a crédito.

---

### `loan_intensity`

Fórmula:

`total_loans / total_accounts`

Essa feature mede a quantidade de empréstimos do cliente em relação ao número de contas.

Exemplo:

- `total_loans = 2`
- `total_accounts = 4`
- `loan_intensity = 0.50`

Isso significa que o cliente possui, em média, 0,5 empréstimos por conta.

Um valor mais alto pode indicar maior uso de produtos de crédito. Dependendo do saldo, do histórico transacional e da taxa de juros, isso também pode indicar maior dependência de crédito.

---

### `high_interest_flag`

Essa feature indica se a taxa média de juros do cliente está acima do percentil 75 entre os clientes que possuem empréstimos.

O percentil 75 é usado como um critério estatístico para identificar o grupo superior da distribuição, ou seja, clientes com taxas de juros relativamente mais altas dentro da própria base.

Isso não é uma regra regulatória nem uma verdade absoluta de mercado. É uma abordagem inicial baseada nos dados, usada quando ainda não existe uma regra de negócio oficial definida.

Clientes com `high_interest_flag = 1` podem estar expostos a custos de crédito mais altos.

Essa feature pode apoiar:

- Financial Health Score
- Análise de risco financeiro
- Recomendações de renegociação
- Alertas no dashboard
- Explicações geradas pela IA

---

### `has_high_credit_exposure`

Essa feature indica se o cliente possui uma exposição elevada a crédito com base no `loan_to_balance_ratio`.

Ou seja, ela ajuda a identificar clientes cujo valor total de empréstimos é alto em relação ao saldo disponível.

Essa feature pode ser usada em:

- Financial Health Score
- Análise de risco financeiro
- Recomendações de renegociação
- Alertas de pressão financeira
- Monitoramento no dashboard

---

### Impacto no projeto

Juntas, essas features transformam dados brutos de empréstimos em indicadores interpretáveis de risco e exposição financeira.

Em vez de analisarmos apenas quanto o cliente pegou emprestado, passamos a entender:

- o tamanho da dívida em relação ao saldo;
- a intensidade de uso de empréstimos;
- se o cliente está exposto a juros mais altos;
- se existe possível pressão financeira;
- se o cliente pode precisar de recomendação, alerta ou ação personalizada.

Essas features serão importantes p
ara a Camada 5 do FinPulse AI, principalmente nos módulos de score financeiro, risco, recomendações e IA generativa.
### Features adicionais para apoiar o Financial Health Score

Além das features específicas de empréstimos, também serão criadas algumas variáveis complementares para apoiar a construção do `financial_health_score`.

Essas features são inspiradas em pilares comuns de saúde financeira, como liquidez, uso de crédito, engajamento financeiro, relacionamento com produtos bancários e resiliência financeira.

Como o dataset não possui informações reais de renda, orçamento mensal, reserva de emergência ou contas pagas, essas variáveis funcionam como aproximações, ou seja, proxies.

O objetivo é transformar os dados disponíveis em sinais úteis para medir o perfil financeiro do cliente.

---

### `relationship_depth`

Essa feature mede a profundidade do relacionamento do cliente com o banco.

Ela soma diferentes vínculos financeiros do cliente, como:

- conta corrente;
- conta poupança;
- conta business;
- cartão;
- empréstimo;
- transações.

Quanto maior o valor, mais produtos e interações o cliente possui com o banco.

Um cliente com maior profundidade de relacionamento tende a ter um vínculo mais forte com a instituição, o que pode impactar positivamente análises de engajamento, churn e saúde financeira.

---

### `product_diversity_score`

Essa feature mede a diversidade de produtos financeiros utilizados pelo cliente.

Ela considera produtos como:

- conta corrente;
- conta poupança;
- conta business;
- cartão de crédito;
- cartão de débito;
- empréstimos.

A diferença para `relationship_depth` é que aqui o foco está mais na variedade de produtos financeiros utilizados, enquanto `relationship_depth` considera também atividade transacional.

Essa feature pode ajudar a identificar clientes mais completos dentro do ecossistema bancário.

---

### `debt_pressure_flag`

Essa feature indica se o cliente apresenta algum sinal de pressão relacionada a crédito.

Ela considera principalmente dois fatores:

- alta exposição a crédito;
- taxa média de juros elevada.

Um cliente marcado com `debt_pressure_flag = 1` pode estar em uma condição de maior atenção financeira, principalmente se possui empréstimos altos em relação ao saldo ou juros acima do padrão da base.

Essa feature será útil para:

- Financial Health Score;
- risco financeiro;
- recomendação de renegociação;
- alertas no dashboard;
- explicações geradas por IA.

---

### `is_financially_active`

Essa feature indica se o cliente está financeiramente ativo.

A lógica considera se o cliente possui transações e se a última transação ocorreu em um período recente.

Clientes ativos tendem a demonstrar maior engajamento financeiro com a instituição.

Essa feature será importante para diferenciar clientes que apenas possuem cadastro ou produtos, mas não apresentam movimentação recente.

---

### `low_engagement_flag`

Essa feature identifica clientes com baixo engajamento.

Ela pode marcar clientes que:

- não possuem transações;
- ou estão há muito tempo sem transacionar.

Essa variável é importante para análises de churn, porque longos períodos de inatividade podem indicar perda de relacionamento com o banco.

Também pode ser usada para campanhas de reativação ou recomendações personalizadas.

---

### `financial_resilience_proxy`

Essa feature cria uma aproximação de resiliência financeira.

Como o dataset não possui informações reais de renda, despesas mensais ou reserva de emergência, usamos sinais disponíveis para estimar uma possível resiliência.

A lógica considera clientes que possuem:

- saldo relativamente alto;
- ausência de pressão elevada de dívida;
- atividade financeira recente.

Clientes com `financial_resilience_proxy = 1` podem representar perfis com maior capacidade financeira aparente dentro da base.

Essa feature não deve ser interpretada como uma medida definitiva de resiliência financeira, mas como um indicador inicial para apoiar o Health Score V1.

---

### Impacto dessas features na Camada 5

Essas features serão usadas como base para a Camada 5 do FinPulse AI.

Elas ajudam principalmente nos seguintes módulos:

- `Financial Health Score`: avaliação geral da saúde financeira do cliente;
- `Credit Risk Analysis`: identificação de pressão e exposição a crédito;
- `Churn Risk`: identificação de clientes com baixo relacionamento ou baixa atividade;
- `Recommendation Engine`: geração de ações personalizadas;
- `AI Assistant`: explicações sobre o perfil financeiro do cliente.

Com essas variáveis, o projeto deixa de analisar apenas dados brutos e passa a criar uma camada de inteligência financeira mais interpretável.

In [40]:
loan_features = [
    "loan_to_balance_ratio",
    "avg_loan_per_account",
    "loan_intensity",
    "high_interest_flag",
    "has_high_credit_exposure"
]

df[loan_features].describe().T

,count,mean,std,min,25%,50%,75%,max
loan_to_balance_ratio,44836.0,1.758628,37.260580,0.0,0.0,0.0,0.715401,5566.426251
avg_loan_per_account,44836.0,51390.389071,95539.637265,0.0,0.0,0.0,71098.180000,978201.470000
loan_intensity,44836.0,0.341553,0.562594,0.0,0.0,0.0,0.500000,6.000000
high_interest_flag,50000.0,0.112880,0.316449,0.0,0.0,0.0,0.000000,1.000000
has_high_credit_exposure,50000.0,0.190400,0.392621,0.0,0.0,0.0,0.000000,1.000000


In [41]:
loan_health_features = [
    "loan_to_balance_ratio",
    "avg_loan_per_account",
    "loan_intensity",
    "high_interest_flag",
    "has_high_credit_exposure",
    "relationship_depth",
    "product_diversity_score",
    "debt_pressure_flag",
    "is_financially_active",
    "low_engagement_flag",
    "financial_resilience_proxy"
]

df[loan_health_features].describe().T

,count,mean,std,min,25%,50%,75%,max
loan_to_balance_ratio,44836.0,1.758628,37.260580,0.0,0.0,0.0,0.715401,5566.426251
avg_loan_per_account,44836.0,51390.389071,95539.637265,0.0,0.0,0.0,71098.180000,978201.470000
loan_intensity,44836.0,0.341553,0.562594,0.0,0.0,0.0,0.500000,6.000000
high_interest_flag,50000.0,0.112880,0.316449,0.0,0.0,0.0,0.000000,1.000000
has_high_credit_exposure,50000.0,0.190400,0.392621,0.0,0.0,0.0,0.000000,1.000000
relationship_depth,50000.0,3.083880,1.648350,0.0,2.0,3.0,4.000000,6.000000
product_diversity_score,50000.0,2.676820,1.621452,0.0,1.0,3.0,4.000000,6.000000
debt_pressure_flag,50000.0,0.256900,0.436928,0.0,0.0,0.0,1.000000,1.000000
is_financially_active,50000.0,0.426880,0.494630,0.0,0.0,0.0,1.000000,1.000000
low_engagement_flag,50000.0,0.378680,0.485063,0.0,0.0,0.0,1.000000,1.000000


In [42]:
df[loan_health_features].isna().sum()

loan_to_balance_ratio         5164
avg_loan_per_account          5164
loan_intensity                5164
high_interest_flag               0
has_high_credit_exposure         0
relationship_depth               0
product_diversity_score          0
debt_pressure_flag               0
is_financially_active            0
low_engagement_flag              0
financial_resilience_proxy       0
dtype: int64

In [43]:
df[
    [
        "high_interest_flag",
        "has_high_credit_exposure",
        "debt_pressure_flag",
        "is_financially_active",
        "low_engagement_flag",
        "financial_resilience_proxy"
    ]
].sum().sort_values(ascending=False)

is_financially_active         21344
low_engagement_flag           18934
debt_pressure_flag            12845
has_high_credit_exposure       9520
financial_resilience_proxy     8049
high_interest_flag             5644
dtype: int64

### Validação das Features de Empréstimos e Saúde Financeira

As features criadas nesta seção foram validadas sem presença de valores nulos.

As variáveis `relationship_depth` e `product_diversity_score` apresentaram valores dentro do intervalo esperado, variando de 0 a 6. Isso confirma que os indicadores de profundidade de relacionamento e diversidade de produtos foram calculados corretamente.

As flags criadas também geraram grupos interpretáveis de clientes, como clientes financeiramente ativos, clientes com baixo engajamento, clientes com pressão de dívida, clientes com alta exposição a crédito e clientes com possível resiliência financeira.

O `loan_to_balance_ratio` apresentou valores máximos elevados. Isso pode acontecer quando o cliente possui saldo baixo e valor de empréstimo relativamente alto. Esse comportamento será mantido nesta etapa por representar um possível sinal de exposição financeira elevada, mas poderá ser tratado posteriormente com técnicas de cap ou transformação para modelagem.

## 9. Transaction Behavior Features

In this step, we create features related to customer transaction behavior.

The original mart already contains aggregated transaction information, such as total transactions, total transaction amount, average transaction amount, active transaction days and merchant diversity.

In this section, we transform these values into more interpretable indicators that help measure transaction frequency, transaction intensity, spending behavior, customer engagement and merchant diversity.

These features will be important for the AI layer of FinPulse AI, especially for:

- Churn Risk
- Financial Health Score
- Financial anomaly detection
- Customer segmentation
- Personalized recommendations
- Dashboard alerts and insights

---

### Features created in this step

---

### `transactions_per_active_day`

Average number of transactions per active transaction day.

Formula:

`total_transactions / active_transaction_days`

This feature helps measure how intensely a customer transacts on the days they are financially active.

A customer may have few active days but many transactions concentrated on those days. This feature helps capture that behavior.

---

### `amount_per_active_day`

Average transaction amount per active transaction day.

Formula:

`total_transaction_amount_usd / active_transaction_days`

This feature helps measure how much money the customer moves on the days they are active.

It can support engagement analysis, customer segmentation and anomaly detection.

---

### `amount_per_transaction`

Average amount per transaction.

Formula:

`total_transaction_amount_usd / total_transactions`

This feature helps measure the customer's average transaction ticket.

Although the mart already contains `avg_transaction_amount_usd`, this feature reinforces the calculation logic and can be used as a consistency check.

---

### `amount_per_merchant`

Average transaction amount per distinct merchant.

Formula:

`total_transaction_amount_usd / merchant_diversity`

This feature helps measure how the customer's transaction volume is distributed across different merchants.

Higher values may indicate concentration of spending in fewer merchants, while lower values may indicate more distributed spending behavior.

---

### `transaction_intensity`

Number of transactions relative to the number of accounts.

Formula:

`total_transactions / total_accounts`

This feature helps normalize transaction activity by the customer's account relationship.

It allows better comparison between customers with different numbers of accounts.

---

### `merchant_diversity_ratio`

Ratio between distinct merchants and total transactions.

Formula:

`merchant_diversity / total_transactions`

This feature measures how diverse the customer's merchant behavior is.

Values closer to 1 indicate that the customer tends to transact with many different merchants, while lower values may indicate repeated transactions with fewer merchants.

---

### Business Impact

These features transform raw transaction aggregates into interpretable behavioral indicators.

With them, we can better understand:

- transaction frequency;
- transaction intensity;
- average transaction value;
- spending distribution;
- customer engagement;
- potential inactivity;
- possible anomalous behavior;
- churn and financial health signals.

These variables will be especially useful for the Financial Health Score, Churn Risk, anomaly detection and recommendation modules.

## 9. Transaction Behavior Features

Nesta etapa, criamos features relacionadas ao comportamento transacional dos clientes.

A mart original já contém informações como quantidade total de transações, valor total transacionado, valor médio de transação, dias ativos e diversidade de estabelecimentos.

Nesta seção, transformamos esses dados em indicadores mais interpretáveis para medir frequência, intensidade, engajamento e comportamento financeiro.

Essas features serão fundamentais para a Camada 5 do FinPulse AI, principalmente nos módulos de:

- Churn Risk
- Financial Health Score
- Detecção de anomalias financeiras
- Segmentação de clientes
- Recomendações personalizadas
- Alertas e insights no dashboard

---

### Features criadas nesta etapa

---

### `transactions_per_active_day`

Mede a média de transações por dia ativo.

Fórmula:

`total_transactions / active_transaction_days`

Essa feature ajuda a entender a intensidade de uso do cliente nos dias em que ele realmente transacionou.

Um cliente pode ter poucas datas ativas, mas muitas transações concentradas nesses dias. Por isso, essa métrica ajuda a entender melhor o comportamento transacional.

---

### `amount_per_active_day`

Mede o valor médio transacionado por dia ativo.

Fórmula:

`total_transaction_amount_usd / active_transaction_days`

Essa feature ajuda a entender o volume financeiro movimentado pelo cliente nos dias em que ele esteve ativo.

Ela pode apoiar análises de engajamento, perfil financeiro e possíveis anomalias.

---

### `amount_per_transaction`

Mede o valor médio transacionado por transação.

Fórmula:

`total_transaction_amount_usd / total_transactions`

Essa feature ajuda a entender o ticket médio transacional do cliente.

Embora a mart já possua `avg_transaction_amount_usd`, essa feature reforça a lógica de cálculo e pode ser usada para validação.

---

### `amount_per_merchant`

Mede o valor médio transacionado por estabelecimento diferente.

Fórmula:

`total_transaction_amount_usd / merchant_diversity`

Essa feature ajuda a entender a concentração ou dispersão do consumo do cliente entre diferentes estabelecimentos.

---

### `transaction_intensity`

Mede a quantidade de transações em relação ao número de contas.

Fórmula:

`total_transactions / total_accounts`

Essa feature ajuda a comparar o comportamento transacional entre clientes com diferentes quantidades de contas.

---

### `merchant_diversity_ratio`

Mede a diversidade de estabelecimentos em relação ao total de transações.

Fórmula:

`merchant_diversity / total_transactions`

Essa feature indica se o cliente transaciona em muitos estabelecimentos diferentes ou se concentra suas transações em poucos lugares.

---

### Impacto no projeto

Essas features ajudam a transformar dados brutos de transações em indicadores de comportamento financeiro.

Com elas, conseguimos avaliar:

- frequência de uso;
- intensidade transacional;
- valor médio movimentado;
- diversidade de consumo;
- possível inatividade;
- possíveis padrões anômalos;
- sinais de churn ou engajamento.

Essas variáveis serão muito importantes para o Financial Health Score, Churn Risk e para os módulos de anomalias e recomendações.

In [44]:
df["transactions_per_active_day"] = np.where(
    df["active_transaction_days"] > 0,
    df["total_transactions"] / df["active_transaction_days"],
    0
)

df["amount_per_active_day"] = np.where(
    df["active_transaction_days"] > 0,
    df["total_transaction_amount_usd"] / df["active_transaction_days"],
    0
)

df["amount_per_transaction"] = np.where(
    df["total_transactions"] > 0,
    df["total_transaction_amount_usd"] / df["total_transactions"],
    0
)

df["amount_per_merchant"] = np.where(
    df["merchant_diversity"] > 0,
    df["total_transaction_amount_usd"] / df["merchant_diversity"],
    0
)

df["transaction_intensity"] = np.where(
    df["total_accounts"] > 0,
    df["total_transactions"] / df["total_accounts"],
    0
)

df["merchant_diversity_ratio"] = np.where(
    df["total_transactions"] > 0,
    df["merchant_diversity"] / df["total_transactions"],
    0
)

In [45]:
transaction_features = [
    "transactions_per_active_day",
    "amount_per_active_day",
    "amount_per_transaction",
    "amount_per_merchant",
    "transaction_intensity",
    "merchant_diversity_ratio"
]

df[transaction_features].describe().T

,count,mean,std,min,25%,50%,75%,max
transactions_per_active_day,50000.0,0.780654,0.418710,0.0,1.000000,1.000000,1.000000,1.222222
amount_per_active_day,50000.0,3903.781405,2176.281091,0.0,3760.700026,4807.899363,5307.321507,8630.715714
amount_per_transaction,50000.0,3884.253564,2164.481272,0.0,3751.159583,4783.941305,5278.411429,8630.715714
amount_per_merchant,50000.0,3894.094575,2170.342447,0.0,3757.834688,4796.132776,5293.912224,8630.715714
transaction_intensity,50000.0,10.352058,6.153009,0.0,8.000000,12.000000,14.500000,31.000000
merchant_diversity_ratio,50000.0,0.774874,0.415498,0.0,0.970588,1.000000,1.000000,1.000000


In [46]:
df[transaction_features].isna().sum()

transactions_per_active_day    0
amount_per_active_day          0
amount_per_transaction         0
amount_per_merchant            0
transaction_intensity          0
merchant_diversity_ratio       0
dtype: int64

In [47]:
df[
    [
        "total_transactions",
        "active_transaction_days",
        "merchant_diversity",
        "transactions_per_active_day",
        "amount_per_transaction",
        "merchant_diversity_ratio"
    ]
].head(10)

,total_transactions,active_transaction_days,merchant_diversity,transactions_per_active_day,amount_per_transaction,merchant_diversity_ratio
0,44,44,44,1.000000,4404.154318,1.000000
1,25,25,25,1.000000,4605.244400,1.000000
2,19,19,19,1.000000,4886.612105,1.000000
3,10,10,10,1.000000,4657.723000,1.000000
4,0,0,0,0.000000,0.000000,0.000000
5,34,34,34,1.000000,5022.272647,1.000000
6,39,39,39,1.000000,5401.247692,1.000000
7,47,45,46,1.044444,4285.280426,0.978723
8,16,16,16,1.000000,5100.391250,1.000000
9,0,0,0,0.000000,0.000000,0.000000


### Transaction Feature Validation

The transaction behavior features were successfully created with no missing values.

The `transactions_per_active_day` feature shows that most customers have close to one transaction per active day, while some customers have more than one transaction on the same day.

The `amount_per_transaction` feature represents the average transaction ticket per customer and is consistent with the original transaction aggregates.

The `transaction_intensity` feature helps normalize transaction activity by the number of accounts, making it easier to compare customers with different account structures.

The `merchant_diversity_ratio` ranges from 0 to 1, confirming that the feature was calculated correctly. Values closer to 1 indicate that the customer tends to transact with many different merchants, while lower values indicate more repeated transactions with the same merchants.

These features will be useful for churn analysis, financial health scoring, anomaly detection and recommendation logic.

### Churn Definition and Feature Rationale

In this project, churn risk is defined as a behavioral indicator of potential customer disengagement.

Since the dataset does not contain a confirmed historical churn label, the objective is not to identify customers who officially closed their accounts. Instead, we create a rule-based churn risk proxy based on inactivity, low engagement and weak relationship signals.

This approach is commonly used as an initial modeling strategy when a true churn label is not available. The resulting `churn_risk_label` should be interpreted as a proxy target for future machine learning experiments, not as confirmed churn.

The churn risk logic is based on behavioral dimensions commonly used in customer retention analysis:

- Recency: how long it has been since the customer last transacted;
- Frequency: how often the customer transacts;
- Monetary behavior: how much the customer moves financially;
- Engagement: whether the customer is still financially active;
- Relationship depth: how many products and interactions the customer has with the institution;
- Product diversity: how integrated the customer is within the financial ecosystem.

The main variables used to define churn risk are:

- `days_since_last_transaction`: captures transaction recency and inactivity;
- `active_transaction_days`: captures frequency of financial activity;
- `total_transactions`: captures transaction usage volume;
- `is_financially_active`: captures recent activity status;
- `low_engagement_flag`: captures inactivity or weak engagement;
- `relationship_depth`: captures depth of relationship with the bank;
- `product_diversity_score`: captures diversity of financial products used;
- `has_card`: captures card relationship and potential recurring usage.

The first version of the churn score is rule-based and interpretable. Customers with higher inactivity, weaker relationship depth and lower product engagement receive higher churn risk scores.

## 10. Churn and Engagement Features

In this step, we create churn and engagement features based on customer activity, relationship depth and product usage.

Since the dataset does not contain a real historical churn label, we create an initial rule-based churn risk score. This score is not treated as confirmed churn, but as a behavioral risk indicator.

The objective is to identify customers who may show signs of disengagement, inactivity or weak relationship with the bank.

These features will support:

- Churn Risk modeling
- Customer engagement analysis
- Retention strategies
- Personalized recommendations
- Dashboard alerts
- AI-generated explanations

---

### Features created in this step

---

### `churn_risk_score`

A rule-based score from 0 to 100 that estimates the customer's risk of churn based on inactivity, low engagement and weak relationship signals.

The score considers factors such as:

- long time since the last transaction;
- low transaction activity;
- low relationship depth;
- low product diversity;
- absence of card relationship;
- absence of financial activity.

Higher values indicate higher churn risk.

---

### `churn_risk_segment`

A categorical segmentation based on the churn risk score.

The segments are:

- `Low`: customers with low churn risk;
- `Medium`: customers with moderate churn risk;
- `High`: customers with high churn risk.

This feature is useful for dashboards, business rules and prioritizing retention actions.

---

### `churn_risk_label`

A binary label created from the churn risk score.

- `0`: low or medium churn risk;
- `1`: high churn risk.

This label can be used as an initial target variable for a future machine learning churn model.

Since this label is rule-based, it should be interpreted as a proxy for churn risk, not as confirmed historical churn.

---

### Business Impact

The churn features help transform customer behavior into actionable retention signals.

With these features, the project can identify customers who may require reactivation campaigns, personalized offers or closer monitoring.

This step connects feature engineering with the future machine learning layer of FinPulse AI.

O que é churn?

Churn é quando um cliente deixa de se relacionar com uma empresa ou reduz fortemente o uso dos seus produtos/serviços.

No contexto bancário/fintech, churn pode aparecer de várias formas:

cliente encerra a conta
cliente para de transacionar
cliente deixa de usar cartão
cliente reduz relacionamento com produtos
cliente migra seu uso financeiro para outra instituição

Como nosso dataset não tem uma coluna real dizendo:

churn = 1

a gente não pode afirmar que o cliente “cancelou”. O correto é criar uma proxy de risco de churn, ou seja, um indicador comportamental de possível abandono.

Então no nosso projeto vamos trabalhar com:

churn_risk_score

e não com churn real confirmado.

A definição honesta para o notebook seria:

In this project, churn risk is defined as a behavioral indicator of potential customer disengagement, based on inactivity, low transaction behavior and weak relationship with financial products.

Por que usar variáveis comportamentais?

Porque a literatura de churn em serviços financeiros usa muito comportamento do cliente: recência, frequência, valor movimentado, engajamento e profundidade de relacionamento.

O conceito de RFM — Recency, Frequency and Monetary Value é muito usado em marketing e retenção para estimar a chance de um cliente voltar a interagir com uma empresa. A lógica é: quanto mais recente, frequente e valiosa é a atividade do cliente, maior tende a ser o vínculo e a chance de continuar ativo.

Também existem estudos de churn em dados financeiros mostrando que variáveis de recência, frequência e valor monetário podem melhorar modelos de churn em serviços financeiros.

Além disso, um estudo do Banco Mundial sobre churn/inatividade em serviços financeiros digitais trata a modelagem de churn como forma de identificar clientes com risco de inatividade e apoiar estratégias de reengajamento.

Variáveis que vamos usar e evidência por trás
1. days_since_last_transaction

Essa é uma das variáveis mais fortes.

Ela mede há quantos dias o cliente fez a última transação.

Por que faz sentido?

Porque churn em banco muitas vezes aparece primeiro como inatividade. O cliente pode não cancelar a conta, mas para de usar. Pela lógica de RFM, a recência é um sinal importante: clientes que interagiram recentemente tendem a ter maior chance de continuar engajados, enquanto clientes há muito tempo sem atividade podem precisar de reativação.

No nosso score:

days_since_last_transaction > 180

significa que o cliente está há mais de seis meses sem transacionar.

Impacto no churn:

quanto maior o tempo sem transacionar, maior o risco de churn comportamental
2. active_transaction_days

Essa variável mede em quantos dias diferentes o cliente transacionou.

Por que faz sentido?

Ela representa frequência de uso. Mesmo que dois clientes tenham transações, existe diferença entre alguém que transacionou em muitos dias diferentes e alguém que transacionou poucas vezes.

A literatura de RFM considera frequência como um dos pilares para prever comportamento futuro do cliente.

Impacto no churn:

menos dias ativos = menor engajamento financeiro
mais dias ativos = maior vínculo comportamental
3. total_transactions

Essa variável mede o volume total de transações do cliente.

Por que faz sentido?

Ela também representa frequência/uso. Estudos de churn em bancos e datasets bancários frequentemente consideram atividade, uso e padrões de transação como variáveis importantes para prever saída ou inatividade.

Impacto no churn:

poucas transações podem indicar baixo uso da instituição
4. total_transaction_amount_usd e amount_per_transaction

Essas variáveis representam o valor monetário movimentado.

Por que faz sentido?

No RFM, o “M” é justamente Monetary Value, ou seja, quanto valor o cliente movimenta/gasta. Isso ajuda a diferenciar um cliente ativo de baixo volume de um cliente ativo de alto valor.

Impacto no churn:

baixo valor movimentado pode indicar menor uso financeiro
alto valor movimentado pode indicar maior vínculo ou maior valor do cliente

No nosso score inicial de churn talvez a gente não coloque peso direto no valor, para não punir cliente de menor renda/saldo. Mas ele pode entrar depois na priorização de retenção.

5. is_financially_active

Essa é uma flag criada por nós.

Ela indica se o cliente:

tem transações
e transacionou recentemente

Por que faz sentido?

Ela resume a ideia de atividade financeira recente. Isso conversa diretamente com estudos que tratam churn como inatividade ou redução de uso em serviços financeiros digitais.

Impacto no churn:

cliente financeiramente ativo = menor risco
cliente inativo = maior risco
6. low_engagement_flag

Essa flag marca clientes com baixo engajamento.

A lógica que propusemos:

sem transações
ou muitos dias sem transacionar

Por que faz sentido?

Porque churn nem sempre começa com cancelamento. Muitas vezes começa com queda de engajamento. O estudo de churn/inatividade em serviços financeiros digitais do Banco Mundial trata a identificação de potenciais churners como caminho para campanhas de reativação e reengajamento.

Impacto no churn:

baixo engajamento = forte sinal de possível churn comportamental
7. relationship_depth

Essa variável mede quantos vínculos o cliente tem com o banco:

conta corrente
conta poupança
conta business
cartão
empréstimo
transações

Por que faz sentido?

Clientes com mais vínculos tendem a ter maior custo de troca e maior integração com a instituição. Estudos de churn bancário frequentemente usam variáveis como número de produtos, atividade e status de membro ativo para explicar saída de clientes. Um estudo sobre churn bancário aponta que crédito, atividade e variáveis de perfil podem afetar a saída; e trabalhos recentes de churn bancário analisam produto, atividade e relacionamento como variáveis preditivas.

Impacto no churn:

menos relacionamento = menor vínculo
mais relacionamento = maior retenção potencial
8. product_diversity_score

Essa variável mede a diversidade de produtos financeiros usados.

Por que faz sentido?

Ela complementa relationship_depth. Em vez de olhar apenas atividade, olha variedade de produtos. Em churn bancário, uso de produtos como cartão, conta ativa e empréstimos pode indicar maior ou menor vínculo com o banco. Estudos de churn em bancos usam variáveis de produto, atividade e engajamento como preditores.

Impacto no churn:

baixa diversidade de produtos = relacionamento fraco
alta diversidade = cliente mais integrado ao ecossistema
9. has_card

Essa flag indica se o cliente possui cartão.

Por que faz sentido?

Cartão costuma gerar uso recorrente e contato frequente com a instituição. Em datasets de churn bancário, variáveis de posse de cartão e atividade aparecem frequentemente como parte da modelagem de churn.

Impacto no churn:

sem cartão = possível menor vínculo transacional
com cartão = maior chance de uso recorrente
Como vamos transformar isso em score?

A ideia é criar um score rule-based de 0 a 100.

Não é “verdade absoluta”. É uma primeira versão interpretável.

A lógica:

inatividade recente pesa mais
baixo engajamento pesa bastante
relacionamento fraco pesa médio
baixa diversidade de produtos pesa médio
ausência de cartão pesa menos
ausência de atividade financeira reforça o risco

Por isso propusemos algo assim:

days_since_last_transaction > 180  → +35 pontos
low_engagement_flag = 1             → +25 pontos
relationship_depth <= 2             → +15 pontos
product_diversity_score <= 2        → +10 pontos
has_card = 0                        → +10 pontos
is_financially_active = 0           → +5 pontos

Total máximo:

100 pontos

Depois segmentamos:

0 a 39   = Low
40 a 69  = Medium
70 a 100 = High

E criamos a label:

High → churn_risk_label = 1
Low/Medium → churn_risk_label = 0

In [48]:
df["churn_heuristic_status"] = np.select(
    [
        df["never_transacted_flag"] == 1,
        df["transaction_timeline_invalid_flag"] == 1
    ],
    [
        "Not Activated",
        "Invalid Timeline"
    ],
    default="Eligible"
)

churn_eligible = (
    df["churn_heuristic_status"] == "Eligible"
)

churn_score = pd.Series(
    0,
    index=df.index,
    dtype="int16"
)

churn_score += np.where(
    df["days_since_last_transaction"] > 365,
    35,
    np.where(
        df["days_since_last_transaction"] > 180,
        25,
        0
    )
)

churn_score += np.where(
    df["active_transaction_days"] <= 5,
    20,
    0
)

churn_score += np.where(
    df["relationship_depth"] <= 1,
    20,
    np.where(
        df["relationship_depth"] <= 2,
        10,
        0
    )
)

churn_score += np.where(
    df["product_diversity_score"] <= 1,
    15,
    np.where(
        df["product_diversity_score"] <= 2,
        5,
        0
    )
)

churn_score += np.where(
    df["has_card"] == 0,
    5,
    0
)

churn_score = churn_score.clip(0, 100)

df["churn_risk_score"] = (
    churn_score.where(churn_eligible)
)

In [49]:
df["churn_risk_segment"] = pd.cut(
    df["churn_risk_score"],
    bins=[-1, 39, 69, 100],
    labels=["Low", "Medium", "High"]
)

In [50]:
df["churn_risk_high_flag"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int8"
)

df.loc[
    churn_eligible,
    "churn_risk_high_flag"
] = (
    df.loc[
        churn_eligible,
        "churn_risk_score"
    ] >= 70
).astype("int8")

In [51]:
churn_features = [
    "churn_heuristic_status",
    "churn_risk_score",
    "churn_risk_segment",
    "churn_risk_high_flag"
]

df[churn_features].head(100)

,churn_heuristic_status,churn_risk_score,churn_risk_segment,churn_risk_high_flag
0,Eligible,0.0,Low,0
1,Eligible,0.0,Low,0
2,Eligible,0.0,Low,0
3,Eligible,25.0,Low,0
4,Not Activated,NaN,NaN,<NA>
...,...,...,...,...
95,Eligible,0.0,Low,0
96,Not Activated,NaN,NaN,<NA>
97,Not Activated,NaN,NaN,<NA>
98,Not Activated,NaN,NaN,<NA>


In [52]:
df[churn_features].isna().sum()

churn_heuristic_status        0
churn_risk_score          13084
churn_risk_segment        13084
churn_risk_high_flag      13084
dtype: int64

In [53]:
df["churn_risk_segment"].value_counts().sort_index()

churn_risk_segment
Low       35208
Medium     1699
High          9
Name: count, dtype: int64

In [54]:
display(
    df["churn_heuristic_status"]
    .value_counts(dropna=False)
    .to_frame("customers")
)

display(
    df.loc[
        churn_eligible,
        "churn_risk_segment"
    ]
    .value_counts()
    .sort_index()
    .to_frame("customers")
)

,customers
churn_heuristic_status,
Eligible,36916
Not Activated,11162
Invalid Timeline,1922


,customers
churn_risk_segment,
Low,35208
Medium,1699
High,9


In [55]:
df.loc[
    df["churn_risk_high_flag"] == 1,
    [
        "customer_id",
        "city",
        "days_since_last_transaction",
        "relationship_depth",
        "product_diversity_score",
        "has_card",
        "churn_risk_score",
        "churn_risk_segment"
    ]
].head(10)

,customer_id,city,days_since_last_transaction,relationship_depth,product_diversity_score,has_card,churn_risk_score,churn_risk_segment
8672,CUSCHH5EU2AZVFX,Brianshire,1206.0,2,1,0,85.0,High
14548,CUSKXYCH6MWDJ1P,Wolfehaven,1735.0,2,1,0,85.0,High
17089,CUSOMK1GNWBV6VN,South Christopherchester,257.0,2,1,0,75.0,High
19683,CUSSGH986GVEYUJ,Jeremyton,481.0,2,1,0,85.0,High
22207,CUSW5W0ZUMK6BJR,South Lauraborough,757.0,2,1,0,85.0,High
25658,CUS14FTNM8VH1XS,South Angela,784.0,2,1,0,85.0,High
39927,CUSLJ55QMDNY3B4,Haleville,192.0,2,1,0,75.0,High
44611,CUSSE7GSQUGRG4M,Fitzpatrickton,627.0,2,1,0,85.0,High
46594,CUSV5TUWPEEYIG2,Hallport,789.0,2,1,0,85.0,High


### Churn Risk Distribution Validation

After recalibrating the churn risk score, the high-risk class became more selective.

The final binary label distribution is:

- `0`: customers with low or medium churn risk
- `1`: customers with high churn risk

The high-risk class represents around 22% of the customer base. This creates a moderately imbalanced classification problem, but not an extreme one.

For future machine learning modeling, accuracy alone will not be enough to evaluate performance. Metrics such as precision, recall, F1-score, ROC-AUC and the confusion matrix will be more appropriate, especially for evaluating the model's ability to identify high-risk customers.

This churn label is still a behavioral proxy, not confirmed historical churn. However, it provides a structured and interpretable target variable for the first version of the churn model.

## 11. Financial Health Score V1

In this step, we create the first version of the Financial Health Score.

The objective is to transform the engineered features into an interpretable score from 0 to 100 that summarizes the customer's financial profile.

This score is not intended to represent an official credit score or a regulatory risk score. Instead, it is a data-driven and explainable financial health indicator created for the FinPulse AI project.

The score is based on five main dimensions:

1. Balance strength  
2. Debt pressure  
3. Transaction engagement  
4. Relationship strength  
5. Financial resilience  

Each dimension contributes up to 20 points, resulting in a final score from 0 to 100.

---

### Score dimensions

---

### `balance_strength_score`

Measures the customer's balance position compared to the customer base.

Customers with higher total balances receive higher scores because they may have stronger apparent liquidity.

This is a proxy, since the dataset does not contain income, expenses or emergency savings information.

---

### `debt_pressure_score`

Measures how much debt pressure the customer may have.

Customers with high credit exposure or high interest rates receive lower scores.

This dimension uses features such as:

- `has_high_credit_exposure`
- `high_interest_flag`

The goal is to penalize customers who may have higher financial pressure due to loans or expensive credit.

---

### `transaction_engagement_score`

Measures how financially active the customer is.

Customers with recent transactions and more active transaction days receive higher scores.

This dimension helps identify customers who are actively using the financial institution.

---

### `relationship_strength_score`

Measures the depth of the customer's relationship with the bank.

Customers with more products and interactions receive higher scores.

This dimension uses the `relationship_depth` feature, which captures accounts, cards, loans and transactions.

---

### `financial_resilience_score`

Measures an initial proxy for financial resilience.

Since the dataset does not include income, expenses or emergency reserve information, this score uses available signals such as:

- balance level;
- absence of debt pressure;
- recent financial activity.

This should be interpreted as an approximation, not a definitive measure of financial resilience.

---

### Final outputs

This step creates two final features:

- `financial_health_score`: numerical score from 0 to 100;
- `financial_health_segment`: customer segment based on the score.

The segments are:

- `Healthy`: customers with stronger financial indicators;
- `Attention`: customers with moderate financial indicators;
- `Critical`: customers with weaker financial indicators.

These outputs will be used in the AI layer, dashboard, recommendation engine and future explanations generated by the assistant.

In [56]:
balance_p25 = df["total_balance_usd"].quantile(0.25)
balance_p50 = df["total_balance_usd"].quantile(0.50)
balance_p75 = df["total_balance_usd"].quantile(0.75)

active_days_p50 = df.loc[
    df["active_transaction_days"] > 0,
    "active_transaction_days"
].quantile(0.50)

active_days_p75 = df.loc[
    df["active_transaction_days"] > 0,
    "active_transaction_days"
].quantile(0.75)

balance_p25, balance_p50, balance_p75, active_days_p50, active_days_p75

(16171.6, 130893.60500000001, 228780.275, 23.0, 33.0)

In [57]:
df["balance_strength_score"] = np.select(
    [
        df["total_balance_usd"] >= balance_p75,
        df["total_balance_usd"] >= balance_p50,
        df["total_balance_usd"] >= balance_p25,
        df["total_balance_usd"] > 0
    ],
    [
        20,
        15,
        10,
        5
    ],
    default=0
)

In [58]:
df["debt_pressure_score"] = 20

df["debt_pressure_score"] -= np.where(
    df["has_high_credit_exposure"] == 1,
    12,
    0
)

df["debt_pressure_score"] -= np.where(
    df["high_interest_flag"] == 1,
    8,
    0
)

df["debt_pressure_score"] = df["debt_pressure_score"].clip(0, 20)

In [59]:
df["transaction_engagement_score"] = 0

df["transaction_engagement_score"] += np.where(
    df["is_financially_active"] == 1,
    10,
    0
)

df["transaction_engagement_score"] += np.select(
    [
        df["active_transaction_days"] >= active_days_p75,
        df["active_transaction_days"] >= active_days_p50,
        df["active_transaction_days"] > 0
    ],
    [
        10,
        7,
        4
    ],
    default=0
)

df["transaction_engagement_score"] = df["transaction_engagement_score"].clip(0, 20)

In [60]:
df["relationship_strength_score"] = (
    df["relationship_depth"] / df["relationship_depth"].max()
) * 20

df["relationship_strength_score"] = df["relationship_strength_score"].round(2)

In [61]:
df["financial_resilience_score"] = 0

df["financial_resilience_score"] += np.where(
    df["total_balance_usd"] >= balance_p50,
    8,
    0
)

df["financial_resilience_score"] += np.where(
    df["debt_pressure_flag"] == 0,
    6,
    0
)

df["financial_resilience_score"] += np.where(
    df["is_financially_active"] == 1,
    6,
    0
)

df["financial_resilience_score"] = df["financial_resilience_score"].clip(0, 20)

In [62]:
df["financial_health_score"] = (
    df["balance_strength_score"] +
    df["debt_pressure_score"] +
    df["transaction_engagement_score"] +
    df["relationship_strength_score"] +
    df["financial_resilience_score"]
)

df["financial_health_score"] = df["financial_health_score"].round(2)

In [63]:
df["financial_health_segment"] = pd.cut(
    df["financial_health_score"],
    bins=[-1, 39, 69, 100],
    labels=["Critical", "Attention", "Healthy"]
)

In [64]:
health_score_features = [
    "balance_strength_score",
    "debt_pressure_score",
    "transaction_engagement_score",
    "relationship_strength_score",
    "financial_resilience_score",
    "financial_health_score",
    "financial_health_segment"
]

df[health_score_features].describe().T

,count,mean,std,min,25%,50%,75%,max
balance_strength_score,50000.0,11.383800,7.233889,0.00,8.75,12.5,16.25,20.0
debt_pressure_score,50000.0,16.812160,5.777995,0.00,12.00,20.0,20.00,20.0
transaction_engagement_score,50000.0,9.177220,7.420126,0.00,4.00,7.0,17.00,20.0
relationship_strength_score,50000.0,10.278915,5.494855,0.00,6.67,10.0,13.33,20.0
financial_resilience_score,50000.0,11.019880,7.032823,0.00,6.00,12.0,20.00,20.0
financial_health_score,50000.0,58.671975,27.394948,3.33,35.33,63.0,82.00,100.0


In [65]:
df[health_score_features].isna().sum()

balance_strength_score          0
debt_pressure_score             0
transaction_engagement_score    0
relationship_strength_score     0
financial_resilience_score      0
financial_health_score          0
financial_health_segment        0
dtype: int64

In [66]:
df["financial_health_segment"].value_counts().sort_index()

financial_health_segment
Critical     13367
Attention    15827
Healthy      20806
Name: count, dtype: int64

In [67]:
df[
    [
        "customer_id",
        "city",
        "total_balance_usd",
        "loan_to_balance_ratio",
        "relationship_depth",
        "active_transaction_days",
        "churn_risk_score",
        "financial_health_score",
        "financial_health_segment"
    ]
].head(10)

,customer_id,city,total_balance_usd,loan_to_balance_ratio,relationship_depth,active_transaction_days,churn_risk_score,financial_health_score,financial_health_segment
0,CUS002V4AVJO5UQ,Hobbston,309600.73,0.589388,5,44,0.0,82.67,Healthy
1,CUS0031SK48C4X5,Lovetown,207173.59,1.220605,5,25,0.0,88.67,Healthy
2,CUS003E82EMAYB3,Port Stephanie,238950.91,0.000000,4,19,0.0,87.33,Healthy
3,CUS0079NNF0ML3Z,Jessemouth,173017.15,0.000000,3,10,25.0,63.00,Attention
4,CUS008083HIW01T,North Mary,0.00,0.000000,0,0,NaN,26.00,Critical
5,CUS00D9DCE5I25S,Deborahville,217603.95,0.000000,4,34,0.0,88.33,Healthy
6,CUS00EX3A3P13EM,East Thomas,160748.90,1.204501,5,39,0.0,77.67,Healthy
7,CUS00FNFWDFF1CF,South Codyborough,441643.01,0.000000,5,45,0.0,96.67,Healthy
8,CUS00FSPOJCW0NC,Georgetown,24169.71,9.379314,3,16,10.0,32.00,Critical
9,CUS00IDX2LVEG7J,East Jacobchester,0.00,NaN,1,0,NaN,3.33,Critical


In [68]:
df.shape

(50000, 90)

### Financial Health Score Validation

The Financial Health Score was successfully created with no missing values.

The final distribution generated three interpretable customer groups:

- `Critical`: customers with weaker financial indicators;
- `Attention`: customers with moderate financial indicators;
- `Healthy`: customers with stronger financial indicators.

The score distribution is balanced enough to support dashboard analysis, customer segmentation and future recommendation logic.

The examples show that customers with higher balances, stronger relationship depth, more transaction activity and lower churn risk tend to receive higher financial health scores.

Customers with no balance, low relationship depth, no recent activity and high churn risk tend to be classified as `Critical`.

This confirms that the Financial Health Score V1 is behaving consistently with the business logic defined for the project.

## 12. Final Feature Dataset

In this step, we create the final feature dataset that will be used for modeling, dashboard analysis, recommendation logic and the AI layer of FinPulse AI.

The original dataframe contains raw mart columns, intermediate features, scores and labels. In this section, we select the most relevant columns and organize them into a clean analytical dataset.

The final dataset includes:

- Customer identification and profile attributes;
- Account and balance features;
- Card relationship features;
- Loan and credit exposure features;
- Transaction behavior features;
- Engagement and churn risk features;
- Financial health score features.

This dataset represents the processed feature layer of the project.

In [69]:
final_feature_columns = [
    # Customer profile
    "customer_id",
    "first_name",
    "last_name",
    "email",
    "city",
    "credit_score",
    "customer_created_at",

    # Original account metrics
    "total_accounts",
    "total_balance_usd",
    "avg_balance_usd",
    "min_balance_usd",
    "max_balance_usd",
    "checking_accounts",
    "savings_accounts",
    "business_accounts",

    # Time-based features
    "customer_record_age_days",
    "days_since_last_transaction",
    "days_until_latest_card_expiration",
    "never_transacted_flag",
    "transaction_timeline_invalid_flag",
    "transaction_recency_valid_flag",

    # Account and balance features
    "balance_per_account",
    "balance_concentration_ratio",
    "has_savings_account",
    "has_business_account",
    "has_checking_account",

    # Card features
    "has_card",
    "total_cards",
    "credit_cards",
    "debit_cards",
    "has_credit_card",
    "has_debit_card",
    "credit_card_ratio",
    "debit_card_ratio",
    "card_product_diversity",

    # Loan and credit exposure
    "has_loan",
    "total_loans",
    "total_loan_amount",
    "avg_loan_amount",
    "avg_interest_rate",
    "loan_to_balance_ratio",
    "debt_without_balance_flag",
    "avg_loan_per_account",
    "loan_intensity",
    "high_interest_flag",
    "has_high_credit_exposure",
    "debt_pressure_flag",

    # Transaction behavior
    "has_transaction",
    "total_transactions",
    "total_transaction_amount_usd",
    "avg_transaction_amount_usd",
    "active_transaction_days",
    "merchant_diversity",
    "transactions_per_active_day",
    "amount_per_active_day",
    "amount_per_transaction",
    "amount_per_merchant",
    "transaction_intensity",
    "merchant_diversity_ratio",

    # Relationship and engagement
    "relationship_depth",
    "product_diversity_score",
    "is_financially_active",
    "low_engagement_flag",
    "financial_resilience_proxy",

    # Heuristic disengagement risk
    "churn_heuristic_status",
    "churn_risk_score",
    "churn_risk_segment",
    "churn_risk_high_flag",

    # Financial Health Score
    "balance_strength_score",
    "debt_pressure_score",
    "transaction_engagement_score",
    "relationship_strength_score",
    "financial_resilience_score",
    "financial_health_score",
    "financial_health_segment"
]

df_features = df[final_feature_columns].copy()

df_features.head()

,customer_id,first_name,last_name,email,city,credit_score,customer_created_at,total_accounts,total_balance_usd,avg_balance_usd,...,churn_risk_score,churn_risk_segment,churn_risk_high_flag,balance_strength_score,debt_pressure_score,transaction_engagement_score,relationship_strength_score,financial_resilience_score,financial_health_score,financial_health_segment
0,CUS002V4AVJO5UQ,Briana,Pugh,lisaspears@example.org,Hobbston,510,2019-12-03,3,309600.73,103200.243333,...,0.0,Low,0,20,12,20,16.67,14,82.67,Healthy
1,CUS0031SK48C4X5,Julie,Harding,greglee@example.org,Lovetown,540,2022-11-23,2,207173.59,103586.795000,...,0.0,Low,0,15,20,17,16.67,20,88.67,Healthy
2,CUS003E82EMAYB3,Lindsey,Hamilton,kburgess@example.com,Port Stephanie,739,2025-09-11,2,238950.91,119475.455000,...,0.0,Low,0,20,20,14,13.33,20,87.33,Healthy
3,CUS0079NNF0ML3Z,Jacqueline,Saunders,renee68@example.org,Jessemouth,678,2023-05-01,1,173017.15,173017.150000,...,25.0,Low,0,15,20,4,10.00,14,63.00,Attention
4,CUS008083HIW01T,Stacey,Walker,moorecheryl@example.com,North Mary,834,2025-10-14,0,0.00,0.000000,...,NaN,NaN,<NA>,0,20,0,0.00,6,26.00,Critical


In [70]:
df_features.shape

(50000, 75)

In [71]:
df_features.duplicated(subset=["customer_id"]).sum()

0

In [72]:
feature_nulls = (
    df_features
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column_name", 0: "null_count"})
)

feature_nulls["null_percentage"] = (
    feature_nulls["null_count"] / len(df_features) * 100
).round(2)

feature_nulls.sort_values("null_count", ascending=False).head(20)

,column_name,null_count,null_percentage
17,days_until_latest_card_expiration,16496,32.99
65,churn_risk_score,13084,26.17
67,churn_risk_high_flag,13084,26.17
16,days_since_last_transaction,13084,26.17
66,churn_risk_segment,13084,26.17
40,loan_to_balance_ratio,5164,10.33
43,loan_intensity,5164,10.33
42,avg_loan_per_account,5164,10.33
45,has_high_credit_exposure,0,0.00
46,debt_pressure_flag,0,0.00


In [73]:
score_summary = df_features[
    [
        "churn_risk_score",
        "financial_health_score",
        "relationship_depth",
        "product_diversity_score",
        "loan_to_balance_ratio",
        "transaction_intensity"
    ]
].describe().T

score_summary

,count,mean,std,min,25%,50%,75%,max
churn_risk_score,36916.0,9.218225,14.762775,0.00,0.00,0.0,25.000000,85.000000
financial_health_score,50000.0,58.671975,27.394948,3.33,35.33,63.0,82.000000,100.000000
relationship_depth,50000.0,3.083880,1.648350,0.00,2.00,3.0,4.000000,6.000000
product_diversity_score,50000.0,2.676820,1.621452,0.00,1.00,3.0,4.000000,6.000000
loan_to_balance_ratio,44836.0,1.758628,37.260580,0.00,0.00,0.0,0.715401,5566.426251
transaction_intensity,50000.0,10.352058,6.153009,0.00,8.00,12.0,14.500000,31.000000


In [74]:
df_features["churn_risk_segment"].value_counts().sort_index()

churn_risk_segment
Low       35208
Medium     1699
High          9
Name: count, dtype: int64

In [75]:
df_features["financial_health_segment"].value_counts().sort_index()

financial_health_segment
Critical     13367
Attention    15827
Healthy      20806
Name: count, dtype: int64

## 13. Export Customer Financial Profile

This dataset represents the current analytical financial profile of each customer.

It is intended for dashboards, financial health analysis, heuristic disengagement indicators, recommendation rules and AI-generated explanations.

This file is not the supervised temporal dataset used to train the future inactivity model. That dataset will be constructed separately using historical cutoff dates and future observation windows.

In [76]:
from pathlib import Path

processed_path = Path("/home/jovyan/data/processed")
processed_path.mkdir(parents=True, exist_ok=True)

output_file = (
    processed_path /
    "customer_financial_profile_features.csv"
)

df_features.to_csv(output_file, index=False)

output_file

PosixPath('/home/jovyan/data/processed/customer_financial_profile_features.csv')

In [77]:
exported_features = pd.read_csv(output_file)

exported_features.head()

,customer_id,first_name,last_name,email,city,credit_score,customer_created_at,total_accounts,total_balance_usd,avg_balance_usd,...,churn_risk_score,churn_risk_segment,churn_risk_high_flag,balance_strength_score,debt_pressure_score,transaction_engagement_score,relationship_strength_score,financial_resilience_score,financial_health_score,financial_health_segment
0,CUS002V4AVJO5UQ,Briana,Pugh,lisaspears@example.org,Hobbston,510,2019-12-03,3,309600.73,103200.243333,...,0.0,Low,0.0,20,12,20,16.67,14,82.67,Healthy
1,CUS0031SK48C4X5,Julie,Harding,greglee@example.org,Lovetown,540,2022-11-23,2,207173.59,103586.795000,...,0.0,Low,0.0,15,20,17,16.67,20,88.67,Healthy
2,CUS003E82EMAYB3,Lindsey,Hamilton,kburgess@example.com,Port Stephanie,739,2025-09-11,2,238950.91,119475.455000,...,0.0,Low,0.0,20,20,14,13.33,20,87.33,Healthy
3,CUS0079NNF0ML3Z,Jacqueline,Saunders,renee68@example.org,Jessemouth,678,2023-05-01,1,173017.15,173017.150000,...,25.0,Low,0.0,15,20,4,10.00,14,63.00,Attention
4,CUS008083HIW01T,Stacey,Walker,moorecheryl@example.com,North Mary,834,2025-10-14,0,0.00,0.000000,...,NaN,NaN,NaN,0,20,0,0.00,6,26.00,Critical


In [78]:
assert df_features.shape[0] == 50000
assert df_features["customer_id"].duplicated().sum() == 0

assert "churn_risk_label" not in df_features.columns
assert "customer_tenure_days" not in df_features.columns
assert "account_relationship_days" not in df_features.columns
assert "days_since_last_account_open" not in df_features.columns
assert "days_since_last_loan" not in df_features.columns

assert (
    df_features["churn_risk_score"].isna().sum()
    ==
    (
        df_features["churn_heuristic_status"] !=
        "Eligible"
    ).sum()
)

assert (
    df_features.loc[
        df_features["debt_without_balance_flag"] == 1,
        "debt_pressure_flag"
    ] == 1
).all()

print("Final feature dataset validation passed!")

Final feature dataset validation passed!
